## Segmentação de clientes `rfm`

In [0]:
# Criar modelo de Machine Learning para Segmentação de Clientes (RFM).
# Algoritmo K-Means Clustering
# ==============================================================================

import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

print("1. Extraindo a Tabela Ouro do Banco de Dados...")
df = spark.table("lh_nautical.lh_nautical_db.gld_lucro_detalhado").toPandas()

# Garantindo que a data seja interpretada corretamente
df['sale_date'] = pd.to_datetime(df['sale_date'])

print("2. Calculando a Matriz RFM (Recência, Frequência, Monetário)...")
# Data da compra mais recente
data_atual = df['sale_date'].max()

rfm = df.groupby('id_client').agg({
    'sale_date': lambda x: (data_atual - x.max()).days, # R: Dias desde a última compra
    'id_sale': 'count',                                 # F: Quantidade de compras
    'gross_profit': 'sum'                               # M: Lucro total gerado
}).reset_index()

# Renomeando as colunas para o padrão internacional
rfm.columns = ['id_client', 'Recency', 'Frequency', 'Monetary']

print("3. Treinando a Inteligência Artificial (K-Means)...")
# Uso do Scaler pra colocar "Dias" e "Dólares" na mesma escala matemática
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm[['Recency', 'Frequency', 'Monetary']])

# Pedimos para a IA criar 3 grupos distintos (clusters)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
rfm['cluster_ia'] = kmeans.fit_predict(rfm_scaled)

print("4. Salvando o resultado como uma nova Tabela Ouro no Databricks...")
# Convertendo o Pandas de volta para Spark para salvar no Banco de Dados
spark_df = spark.createDataFrame(rfm)
spark_df.write.mode("overwrite").saveAsTable("lh_nautical.lh_nautical_db.gld_segmentacao_clientes")

print("SUCESSO! Tabela 'gld_segmentacao_clientes' criada. Veja a amostra:")
display(spark_df.limit(10))

1. Extraindo a Tabela Ouro do Banco de Dados...
2. Calculando a Matriz RFM (Recência, Frequência, Monetário)...
3. Treinando a Inteligência Artificial (K-Means)...
4. Salvando o resultado como uma nova Tabela Ouro no Databricks...
SUCESSO! Tabela 'gld_segmentacao_clientes' criada. Veja a amostra:


id_client,Recency,Frequency,Monetary,cluster_ia
1,0,190,4.079657615E7,2
2,0,220,5.2255157910000004E7,0
3,16,207,4.702343633E7,1
4,1,207,4.035883873E7,0
5,0,202,4.66676252E7,0
6,2,201,4.218919517E7,0
7,0,207,3.797054997E7,0
8,1,208,4.170422932E7,0
9,1,218,5.326339342E7,0
10,0,203,4.148211999E7,0


## Separação em grupos ( clusters)

In [0]:
print("--- CLUSTERS DA IA ---")

# Vamos agrupar pela coluna 'cluster_ia' e calcular a média das outras colunas
perfil_clusters = rfm.groupby('cluster_ia').agg({
    'Recency': 'mean',     # Média de dias sem comprar
    'Frequency': 'mean',   # Média de compras feitas
    'Monetary': 'mean',    # Média de lucro gerado
    'id_client': 'count'   # Quantos clientes caíram dentro desta caixa?
}).round(2)

# Renomeando as colunas para ficar bonito
perfil_clusters.columns = ['Média_Dias_Ausente', 'Média_Compras', 'Média_Lucro_USD', 'Total_Clientes']

display(perfil_clusters)

--- O RAIO-X DOS CLUSTERS DA IA ---


Média_Dias_Ausente,Média_Compras,Média_Lucro_USD,Total_Clientes
0.92,210.38,4.474878074E7,24
7.09,201.36,4.541917123E7,11
2.43,187.93,3.627422183E7,14


## Previsão e recomendação

In [0]:
print("Linguagem de Negócios...")

def traduzir_cluster(numero_caixa):
    if numero_caixa == 0:
        return 'VIPs (Campeões)'
    elif numero_caixa == 2:
        return 'Em Risco (Adormecidos)'
    else:
        return 'Clientes Regulares'

rfm['Business_Profile'] = rfm['cluster_ia'].apply(traduzir_cluster)

def recomendar_acao(perfil):
    if perfil == 'VIPs (Campeões)':
        return 'Oferecer acesso antecipado a novos produtos VIP.'
    elif perfil == 'Em Risco (Adormecidos)':
        return 'Enviar cupom de desconto agressivo para reativação.'
    else:
        return 'Enviar campanhas de produtos complementares'

rfm['Marketing_Action'] = rfm['Business_Profile'].apply(recomendar_acao)

print("2. Salvando a Tabela Definitiva com as Recomendações...")
spark_df_final = spark.createDataFrame(rfm)

spark_df_final.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("lh_nautical.lh_nautical_db.gld_segmentacao_clientes")

print("SUCESSO! Tabela Finalizada.")
display(spark_df_final.select('id_client', 'Recency', 'Frequency', 'Monetary', 'Business_Profile', 'Marketing_Action').limit(10))

Linguagem de Negócios...
2. Salvando a Tabela Definitiva com as Recomendações...
SUCESSO! Tabela Finalizada.


id_client,Recency,Frequency,Monetary,Business_Profile,Marketing_Action
1,0,190,4.079657615E7,Em Risco (Adormecidos),Enviar cupom de desconto agressivo para reativação.
2,0,220,5.2255157910000004E7,VIPs (Campeões),Oferecer acesso antecipado a novos produtos VIP.
3,16,207,4.702343633E7,Clientes Regulares,Enviar campanhas de produtos complementares
4,1,207,4.035883873E7,VIPs (Campeões),Oferecer acesso antecipado a novos produtos VIP.
5,0,202,4.66676252E7,VIPs (Campeões),Oferecer acesso antecipado a novos produtos VIP.
6,2,201,4.218919517E7,VIPs (Campeões),Oferecer acesso antecipado a novos produtos VIP.
7,0,207,3.797054997E7,VIPs (Campeões),Oferecer acesso antecipado a novos produtos VIP.
8,1,208,4.170422932E7,VIPs (Campeões),Oferecer acesso antecipado a novos produtos VIP.
9,1,218,5.326339342E7,VIPs (Campeões),Oferecer acesso antecipado a novos produtos VIP.
10,0,203,4.148211999E7,VIPs (Campeões),Oferecer acesso antecipado a novos produtos VIP.
